# Hirriririir Multimodal Thigh Segmentation

Runs the SegResNetDS model from [Hirriririir/Multimodal-Multiethnic-Thigh-Muscle-MRI-analysis](https://github.com/Hirriririir/Multimodal-Multiethnic-Thigh-Muscle-MRI-analysis)
on the fat-fraction stacks.

The model accepts a **single-channel** NIfTI input — fat-fraction, water, or fat images all work.
It segments 11 thigh muscles (no L/R distinction; assumes single-leg or bilateral FOV).

**Setup:** download `pretrained_segmentation_muscle.pt` from the GitHub releases (see cell 3).

Output: `multimodal_thigh_segs/*_thigh_seg.nii.gz` in original image space.

**Kernel:** `dafne_clean`

In [1]:
import glob
import os
import numpy as np
import torch
import SimpleITK as sitk
from monai.networks.nets import SegResNetDS
from monai.inferers import sliding_window_inference

C:\Users\docto\miniconda3\envs\dafne_clean\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
# Download checkpoint if not present:
# https://github.com/Hirriririir/Multimodal-Multiethnic-Thigh-Muscle-MRI-analysis/releases/tag/1.0
# File: pretrained_segmentation_muscle.pt

CHECKPOINT  = r"C:\Projects\dissector\eval_notebooks\pretrained_segmentation_muscle.pt"
EVAL_DIR    = r"C:\Projects\dissector\eval_notebooks"
IMAGE_GLOB  = os.path.join(EVAL_DIR, "myosegmenTUM", "*", "ImageData",
                           "*FATFRACTION", "*FATFRACTION_stack*.nii")
OUTPUT_DIR  = os.path.join(EVAL_DIR, "multimodal_thigh_segs")
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

TARGET_SPACING = (0.7813, 0.7813, 4.0)   # mm, as trained
ROI_SIZE       = [336, 336, 88]           # sliding window size

LABEL_MAP = {
    1:  "Sartorius",
    2:  "Rectus_Femoris",
    3:  "Vastus_Lateralis",
    4:  "Vastus_Intermedius",
    5:  "Vastus_Medialis",
    6:  "Adductor_Magnus",
    7:  "Gracilis",
    8:  "Biceps_Femoris_Long",
    9:  "Semitendinosus",
    10: "Semimembranosus",
    11: "Biceps_Femoris_Short",
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Device:", DEVICE)

Device: cpu


In [3]:
model = SegResNetDS(
    spatial_dims=3,
    in_channels=1,
    out_channels=12,
    init_filters=32,
    blocks_down=(1, 2, 2, 4, 4),
    dsdepth=4,
    norm="INSTANCE",
    resolution=TARGET_SPACING,
)

ckpt = torch.load(CHECKPOINT, map_location="cpu")
# Handle various checkpoint key conventions
if isinstance(ckpt, dict):
    state = ckpt.get("state_dict") or ckpt.get("network_weights") or ckpt
else:
    state = ckpt
missing, unexpected = model.load_state_dict(state, strict=False)
if missing:    print("Missing keys:",    missing[:5])
if unexpected: print("Unexpected keys:", unexpected[:5])

model = model.to(DEVICE).eval()
print("Model loaded on", DEVICE)

Model loaded on cpu


In [4]:
def resample_sitk(sitk_img, new_spacing, interpolator=sitk.sitkLinear):
    orig_spacing = sitk_img.GetSpacing()          # (x, y, z)
    orig_size    = sitk_img.GetSize()             # (x, y, z)
    new_size = [
        int(round(orig_size[i] * orig_spacing[i] / new_spacing[i]))
        for i in range(3)
    ]
    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(new_spacing)
    resampler.SetSize(new_size)
    resampler.SetOutputDirection(sitk_img.GetDirection())
    resampler.SetOutputOrigin(sitk_img.GetOrigin())
    resampler.SetTransform(sitk.Transform())
    resampler.SetDefaultPixelValue(0)
    resampler.SetInterpolator(interpolator)
    return resampler.Execute(sitk_img)


def preprocess(nii_path):
    img_sitk  = sitk.ReadImage(nii_path, sitk.sitkFloat32)
    resampled = resample_sitk(img_sitk, TARGET_SPACING, sitk.sitkLinear)
    arr       = sitk.GetArrayFromImage(resampled).astype(np.float32)  # (D, H, W)
    mask      = arr > 0
    if mask.any():
        arr[mask] = (arr[mask] - arr[mask].mean()) / (arr[mask].std() + 1e-8)
    return arr, resampled, img_sitk


def infer_volume(arr):
    tensor = torch.tensor(arr[None, None]).float().to(DEVICE)   # (1, 1, D, H, W)
    with torch.no_grad():
        out = sliding_window_inference(
            tensor, roi_size=ROI_SIZE, sw_batch_size=1,
            predictor=model, overlap=0.5, mode="gaussian"
        )
    # SegResNetDS returns list of deep-supervision outputs; take the full-res one
    logits = out[0] if isinstance(out, (list, tuple)) else out
    return torch.argmax(logits, dim=1).squeeze(0).cpu().numpy().astype(np.uint8)

In [5]:
image_files = sorted(glob.glob(IMAGE_GLOB))
print(f"Found {len(image_files)} fat-fraction stacks")

Found 54 fat-fraction stacks


In [ ]:
for nii_path in image_files:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    out_path = os.path.join(OUTPUT_DIR, f"{stem}_thigh_seg.nii.gz")

    if os.path.exists(out_path):
        print(f"Skipping (done): {out_path}")
        continue

    print(f"\nProcessing: {nii_path}")
    arr, resampled_ref, orig_sitk = preprocess(nii_path)
    print(f"  Resampled shape (D,H,W): {arr.shape}")

    pred_resampled = infer_volume(arr)               # (D, H, W) uint8 at target spacing
    print(f"  Labels present: {sorted(np.unique(pred_resampled).tolist())}")

    # Place prediction back into target-spaced image and resample to original space
    pred_sitk = sitk.GetImageFromArray(pred_resampled)
    pred_sitk.CopyInformation(resampled_ref)
    pred_orig = resample_sitk(pred_sitk, orig_sitk.GetSpacing(), sitk.sitkNearestNeighbor)

    # Crop/pad to match original size exactly
    orig_size = orig_sitk.GetSize()
    pred_orig = sitk.Resample(pred_orig, orig_sitk,
                               sitk.Transform(), sitk.sitkNearestNeighbor, 0)

    sitk.WriteImage(pred_orig, out_path)
    print(f"  Saved -> {out_path}")

    # Per-file voxel summary
    pred_arr = sitk.GetArrayFromImage(pred_orig)
    print(f"  {'Label':<6} {'Muscle':<25} {'Voxels':>10}")
    print(f"  {'-'*45}")
    for label_idx, name in LABEL_MAP.items():
        n = int((pred_arr == label_idx).sum())
        if n > 0:
            print(f"  {label_idx:<6} {name:<25} {n:>10,}")

print("\nAll done.")


Processing: C:\Projects\dissector\eval_notebooks\myosegmenTUM\HV001_1\ImageData\HV001_1_FATFRACTION\HV001_1_FATFRACTION_stack1.nii
  Resampled shape (D,H,W): (65, 860, 860)


C:\Users\docto\miniconda3\envs\dafne_clean\lib\site-packages\monai\inferers\utils.py:231: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\python_variable_indexing.cpp:353.)
  win_data = inputs[unravel_slice[0]].to(sw_device)


In [ ]:
# Sanity check
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.nii.gz")))
if results:
    sample = sitk.ReadImage(results[0])
    arr    = sitk.GetArrayFromImage(sample)
    print("Sample:", results[0])
    print("  Shape:", arr.shape, " Spacing:", sample.GetSpacing())
    print("  Labels:", sorted(np.unique(arr).tolist()))
else:
    print("No results yet.")